In [ ]:
from config import *
from models import *

from SMPyBandits.Arms import Bernoulli
from SMPyBandits.Environment import MAB
from SMPyBandits.Policies import Thompson, UCB
from SMPyBandits.Policies.Posterior import Beta

#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
# initialize training models and optimizers
DisRNN_critic = torch.nn.Linear(DisRNN_hidden_size, 1).to(device)
DisRNN_optimizer = torch.optim.Adam(
    list(DisRNN.parameters()) + list(DisRNN_critic.parameters()), 
    lr= 5e-3
)

LSTM_critic = torch.nn.Linear(LSTM_hidden_size, 1).to(device)
LSTM_optimizer = torch.optim.Adam(
    list(LSTM.parameters()) + list(LSTM_readout.parameters()) + list(LSTM_critic.parameters()), 
    lr= 1e-3
)

In [ ]:
# training hyperparameters
batch_size = 32
batch_idx = torch.arange(batch_size, device= device)
beta_e = 1.0
anneal_end = 2000
beta_v = 0.05
beta_max = 1e-3
warmup_start = 0
warmup_end = 10_000

In [ ]:
# load checkpoint
episode = 49999
episodes = 60000
checkpoint = torch.load(f'checkpoints/seed{seed}/checkpoint_ep{episode}.pt')
# checkpoint = torch.load(f'/content/drive/MyDrive/checkpoints/seed{seed}/checkpoint_ep{episode}.pt')

DisRNN.load_state_dict(checkpoint['DisRNN_state_dict'])
DisRNN_critic.load_state_dict(checkpoint['DisRNN_critic_state_dict'])
DisRNN_optimizer.load_state_dict(checkpoint['DisRNN_optimizer_state_dict'])

LSTM.load_state_dict(checkpoint['LSTM_state_dict'])
LSTM_readout.load_state_dict(checkpoint['LSTM_readout_state_dict'])
LSTM_critic.load_state_dict(checkpoint['LSTM_critic_state_dict'])
LSTM_optimizer.load_state_dict(checkpoint['LSTM_optimizer_state_dict'])

In [ ]:
# training
DisRNN_regret_history = []
LSTM_regret_history = []
for ep in range(episodes):
    # sample task
    probs = D(batch_size, num_arms, device= device)


    # reset DisRNN state
    DisRNN.train()
    DisRNN_optimizer.zero_grad()

    DisRNN_h = torch.zeros(batch_size, DisRNN_hidden_size, device= device)
    DisRNN_x = torch.zeros(batch_size, input_size, device= device)

    DisRNN_log_probs = []
    DisRNN_rewards = []
    DisRNN_expected_rewards = []
    DisRNN_entropies = []
    DisRNN_bottleneck_losses = {'h': [], 'x': [], 'z': []}
    DisRNN_regrets = []

    # reset LSTM state
    LSTM.train()
    LSTM_optimizer.zero_grad()

    LSTM_h = torch.zeros(1, batch_size, LSTM_hidden_size, device= device)
    LSTM_c = torch.zeros(1, batch_size, LSTM_hidden_size, device= device)
    LSTM_x = torch.zeros(batch_size, input_size, device= device)

    LSTM_log_probs = []
    LSTM_rewards = []
    LSTM_expected_rewards = []
    LSTM_entropies = []
    LSTM_regrets = []


    for t in range(T):
        if t % 50 == 0:
            DisRNN_h = DisRNN_h.detach()
            LSTM_h = LSTM_h.detach()
            LSTM_c = LSTM_c.detach()


        # DisRNN step
        DisRNN_h, kls = DisRNN.step(DisRNN_h, DisRNN_x)
        DisRNN_logits = DisRNN.out(DisRNN_h)

        DisRNN_pi = torch.distributions.Categorical(logits= DisRNN_logits)
        DisRNN_a = DisRNN_pi.sample()
        DisRNN_r = (torch.rand(batch_size, device= device) < probs[batch_idx, DisRNN_a]).float()

        DisRNN_log_probs.append(DisRNN_pi.log_prob(DisRNN_a))
        DisRNN_rewards.append(DisRNN_r)
        DisRNN_expected_rewards.append(DisRNN_critic(DisRNN_h.detach()).squeeze(-1))
        DisRNN_entropies.append(DisRNN_pi.entropy())
        for key, val in kls.items():
            DisRNN_bottleneck_losses[key].append(val)
        DisRNN_regrets.append(probs.max(dim= -1).values - probs[batch_idx, DisRNN_a])

        DisRNN_x = torch.stack([2*DisRNN_a.float() - 1, 2*DisRNN_r - 1], dim= -1)


        # LSTM step
        LSTM_out, (LSTM_h, LSTM_c) = LSTM(LSTM_x.unsqueeze(0), (LSTM_h, LSTM_c))
        LSTM_logits = LSTM_readout(LSTM_out.squeeze(0))

        LSTM_pi = torch.distributions.Categorical(logits= LSTM_logits)
        LSTM_a = LSTM_pi.sample()
        LSTM_r = (torch.rand(batch_size, device= device) < probs[batch_idx, LSTM_a]).float()
        
        LSTM_log_probs.append(LSTM_pi.log_prob(LSTM_a))
        LSTM_rewards.append(LSTM_r)
        LSTM_expected_rewards.append(LSTM_critic(LSTM_out.squeeze(0).detach()).squeeze(-1))
        LSTM_entropies.append(LSTM_pi.entropy())
        LSTM_regrets.append(probs.max(dim= -1).values - probs[batch_idx, LSTM_a])

        LSTM_x = torch.stack([2*LSTM_a.float() - 1, 2*LSTM_r - 1], dim= -1)
        

    DisRNN_log_probs = torch.stack(DisRNN_log_probs)
    DisRNN_rewards = torch.stack(DisRNN_rewards)
    DisRNN_expected_rewards = torch.stack(DisRNN_expected_rewards)
    DisRNN_entropies = torch.stack(DisRNN_entropies)
    DisRNN_bottleneck_losses = {key: torch.stack(vals) for key, vals in DisRNN_bottleneck_losses.items()}
    DisRNN_regrets = torch.stack(DisRNN_regrets)
    DisRNN_regret_history.append(DisRNN_regrets.mean().item())

    LSTM_log_probs = torch.stack(LSTM_log_probs)
    LSTM_rewards = torch.stack(LSTM_rewards)
    LSTM_expected_rewards = torch.stack(LSTM_expected_rewards)
    LSTM_entropies = torch.stack(LSTM_entropies)
    LSTM_regrets = torch.stack(LSTM_regrets)
    LSTM_regret_history.append(LSTM_regrets.mean().item())


    # update betas
    beta_e = max(0.0, 1.0 - ep / anneal_end)
    beta = beta_max * min((ep - warmup_start) / (warmup_end - warmup_start), 1.0)
    
    # advantage actor-critic
    DisRNN_advantage = DisRNN_rewards - DisRNN_expected_rewards
    DisRNN_loss_actor = -(DisRNN_log_probs * DisRNN_advantage.detach()).mean()
    DisRNN_loss_critic = F.mse_loss(DisRNN_expected_rewards, DisRNN_rewards)
    DisRNN_loss_entropy = DisRNN_entropies.mean()
    DisRNN_loss_bottlenecks = sum(loss.mean() for loss in DisRNN_bottleneck_losses.values())
    DisRNN_loss = (
        DisRNN_loss_actor 
        + beta_v * DisRNN_loss_critic
        - beta_e * DisRNN_loss_entropy 
        + beta * DisRNN_loss_bottlenecks
    )
    
    DisRNN_loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(DisRNN.parameters()) + list(DisRNN_critic.parameters()),
        max_norm= 1.0
    )
    DisRNN_optimizer.step()

    LSTM_advantage = LSTM_rewards - LSTM_expected_rewards
    LSTM_loss_actor = -(LSTM_log_probs * LSTM_advantage.detach()).mean()
    LSTM_loss_critic = F.mse_loss(LSTM_expected_rewards, LSTM_rewards)
    LSTM_loss_entropy = LSTM_entropies.mean()
    LSTM_loss = (
        LSTM_loss_actor
        + beta_v * LSTM_loss_critic
        - beta_e * LSTM_loss_entropy
    )

    LSTM_loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(LSTM.parameters()) + list(LSTM_readout.parameters()) + list(LSTM_critic.parameters()),
        max_norm= 1.0
    )
    LSTM_optimizer.step()

    
    if ep % 250 == 0:
        DisRNN_avg_reward = DisRNN_rewards.mean().item()
        LSTM_avg_reward = LSTM_rewards.mean().item()
        print(f'ep {ep:5d}')
        print(f'LSTM avg r {LSTM_avg_reward:.3f} | DisRNN avg r {DisRNN_avg_reward:.3f}')
        M_h = torch.sigmoid(DisRNN.logit_M_h).detach().cpu().numpy()
        M_x = torch.sigmoid(DisRNN.logit_M_x).detach().cpu().numpy()
        M_z = torch.sigmoid(DisRNN.logit_M_z).detach().cpu().numpy()
        print()
        print(format_matrix(M_h, 'M_h', row_prefix= 'rule', col_prefix= 'lat'))
        print()
        print(format_matrix(M_x, 'M_x', row_prefix= 'rule', col_prefix= 'obs'))
        print()
        print(format_matrix(M_z.reshape(1,-1), 'M_z', row_prefix= 'lat', col_prefix= 'lat'))
        print()
        print()

    if (ep % 5000 == 0 and ep > 0) or ep == episodes - 1:
        torch.save({
            'ep': ep,
            'DisRNN_state_dict': DisRNN.state_dict(),
            'DisRNN_critic_state_dict': DisRNN_critic.state_dict(),
            'DisRNN_optimizer_state_dict': DisRNN_optimizer.state_dict(),
            'LSTM_state_dict': LSTM.state_dict(),
            'LSTM_readout_state_dict': LSTM_readout.state_dict(),
            'LSTM_critic_state_dict': LSTM_critic.state_dict(),
            'LSTM_optimizer_state_dict': LSTM_optimizer.state_dict(),
        }, f'checkpoints/seed{seed}/checkpoint_ep{ep}.pt') # f'/content/drive/MyDrive/checkpoints/seed{seed}/checkpoint_ep{ep}.pt')

        plt.figure(figsize= (8,5))
        plt.plot(smooth(np.array(DisRNN_regret_history)), label= 'DisRNN', color= 'green')
        plt.plot(smooth(np.array(LSTM_regret_history)), label= 'LSTM', color= 'black')
        plt.xlabel('Episode')
        plt.ylabel('Regret')
        plt.title('Model Regret Over Time')
        plt.legend()
        plt.savefig(f'figs/seed{seed}/regret_ep{ep}.png')
        # plt.savefig(f'/content/drive/MyDrive/figs/seed{seed}/regret_ep{ep}.png')
        plt.close()

In [ ]:
# load trained models
checkpoint = torch.load(f'checkpoints/seed{seed}/checkpoint_ep{episodes - 1}.pt')
# checkpoint = torch.load(f'/content/drive/MyDrive/checkpoints/seed{seed}/checkpoint_ep{episode}.pt')
DisRNN.load_state_dict(checkpoint['DisRNN_state_dict'])
LSTM.load_state_dict(checkpoint['LSTM_state_dict'])
LSTM_readout.load_state_dict(checkpoint['LSTM_readout_state_dict'])

In [ ]:
# testing
num_tests = 150

DisRNN_cumulative_regrets = []
LSTM_cumulative_regrets = []
Thompson_cumulative_regrets = []
UCB_cumulative_regrets = []
for _ in range(num_tests):
    p = D(num_arms)
    probs = p.unsqueeze(0).to(device)
    env = MAB([Bernoulli(p[i].item()) for i in range(num_arms)])
    

    # reset DisRNN state
    DisRNN.eval()
    DisRNN_h = torch.zeros(1, DisRNN_hidden_size, device= device)
    DisRNN_x = torch.zeros(1, input_size, device= device)

    # reset LSTM state
    LSTM.eval()
    LSTM_h = torch.zeros(1, 1, LSTM_hidden_size, device= device)
    LSTM_c = torch.zeros(1, 1, LSTM_hidden_size, device= device)
    LSTM_x = torch.zeros(1, input_size, device= device)

    # test models
    thompson = Thompson(num_arms, Beta)
    ucb = UCB(num_arms)


    DisRNN_regrets = []
    LSTM_regrets = []
    thompson_regrets = []
    ucb_regrets = []
    with torch.no_grad():
        for t in range(T):
            optimal = probs.max(dim= -1).values

            # DisRNN step
            DisRNN_h, kls = DisRNN.step(DisRNN_h, DisRNN_x)
            DisRNN_logits = DisRNN.out(DisRNN_h)

            DisRNN_pi = torch.distributions.Categorical(logits= DisRNN_logits)
            DisRNN_a = DisRNN_pi.sample()
            DisRNN_r = (torch.rand(1, device= device) < probs[0, DisRNN_a]).float()
            DisRNN_x = torch.stack([2*DisRNN_a.float() - 1, 2*DisRNN_r - 1], dim= -1)
            DisRNN_regrets.append((optimal - probs[0, DisRNN_a]).cpu())

            # LSTM step
            LSTM_out, (LSTM_h, LSTM_c) = LSTM(LSTM_x.unsqueeze(0), (LSTM_h, LSTM_c))
            LSTM_logits = LSTM_readout(LSTM_out.squeeze(0))

            LSTM_pi = torch.distributions.Categorical(logits= LSTM_logits)
            LSTM_a = LSTM_pi.sample()
            LSTM_r = (torch.rand(1, device= device) < probs[0, LSTM_a]).float()
            LSTM_x = torch.stack([2*LSTM_a.float() - 1, 2*LSTM_r - 1], dim= -1)
            LSTM_regrets.append((optimal - probs[0, LSTM_a]).cpu())

            # Thompson step
            thompson_a = thompson.choice()
            thompson_r = env.draw(thompson_a)
            thompson.getReward(thompson_a, thompson_r)
            thompson_regrets.append(optimal.item() - p[thompson_a].item())

            # UCB step
            ucb_a = ucb.choice()
            ucb_r = env.draw(ucb_a)
            ucb.getReward(ucb_a, ucb_r)
            ucb_regrets.append(optimal.item() - p[ucb_a].item())
            
            
    DisRNN_cumulative_regrets.append(np.array(DisRNN_regrets).cumsum())
    LSTM_cumulative_regrets.append(np.array(LSTM_regrets).cumsum())
    Thompson_cumulative_regrets.append(np.array(thompson_regrets).cumsum())
    UCB_cumulative_regrets.append(np.array(ucb_regrets).cumsum())

In [ ]:
def plot_agent(data, color, label):
    mean = np.stack(data).mean(axis= 0)
    std = np.stack(data).std(axis= 0)
    plt.plot(mean, color= color, label= label)
    plt.fill_between(range(T), mean - std, mean + std, alpha= 0.1, color= color)

plt.figure(figsize= (8,5))
plot_agent(DisRNN_cumulative_regrets, 'green', 'DisRNN')
plot_agent(LSTM_cumulative_regrets, 'black', 'LSTM')
plot_agent(Thompson_cumulative_regrets, 'gray', 'Thompson')
plot_agent(UCB_cumulative_regrets, 'blue', 'UCB')
plt.xlabel('Trial')
plt.ylabel('Cumulative Regret')
plt.title('Model Cumulative Regret')
plt.legend()
plt.grid()
plt.savefig(f'figs/seed{seed}/cumulative_regret.png')
plt.close()